In [ ]:
import json
import pandas as pd

files = {
    "Wav2Vec2": "outputs/fs2_pre_switch_validation.json",
    "HuBERT": "outputs/hubert_pre_switch_validation.json",
    "WavLM": "outputs/wavlm_pre_switch_validation.json",
    "XLS-R-1": "outputs/xls-r_1_pre_switch_validation.json",
    "XLS-R-2": "outputs/xls-r_2_pre_switch_validation.json"
}

delta_sec = [0.05, 0.10, 0.20, 0.50, 1.00, 2.00, 5.00]
xlsr1_deltas = [0.05, 0.10, 0.20, 0.50]
xlsr2_deltas = [1.00, 2.00, 5.00]

results = {}

for model, path in files.items():
    with open(path, "r") as f:
        results[model] = json.load(f)

xlsr = {}

for delta in xlsr1_deltas:
    delta_key = str(delta)
    if delta_key in results["XLS-R-1"]:
        xlsr[delta_key] = results["XLS-R-1"][delta_key]

for delta in xlsr2_deltas:
    delta_key = str(delta)
    if delta_key in results["XLS-R-2"]:
        xlsr[delta_key] = results["XLS-R-2"][delta_key]

results["XLS-R"] = xlsr

del results["XLS-R-1"]
del results["XLS-R-2"]

num_layers = {
    "Wav2Vec2": 13,
    "HuBERT": 13,
    "WavLM": 13,
    "XLS-R": 25
}

rows = []

for model, data in results.items():

    for layer in range(num_layers[model]):

        roc_auc = []
        ap = []
        log_loss = []
        brier = []

        for delta in delta_sec:

            delta_key = str(delta)
            layer_key = str(layer)

            if delta_key in data and layer_key in data[delta_key]:

                metrics = data[delta_key][layer_key]["metrics"]

                roc_auc.append(metrics["roc_auc"])
                ap.append(metrics["ap"])
                log_loss.append(metrics["log_loss"])
                brier.append(metrics["brier_score"])

        if len(roc_auc) == len(ap) == len(log_loss) == len(brier) == len(delta_sec):

            rows.append({
                "SSL Model": model,
                "Layer": layer,
                "Mean ROC-AUC": sum(roc_auc) / len(roc_auc),
                "Mean AP": sum(ap) / len(ap),
                "Mean Log Loss": sum(log_loss) / len(log_loss),
                "Mean Brier Score": sum(brier) / len(brier)
            })

aggregated_results = pd.DataFrame(rows)

print("Number of layers included")
print(aggregated_results.groupby("SSL Model").size())

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

roc_auc_ranking = aggregated_results.sort_values("Mean ROC-AUC", ascending=False).reset_index(drop=True)
ap_ranking = aggregated_results.sort_values("Mean AP", ascending=False).reset_index(drop=True)
log_loss_ranking = aggregated_results.sort_values("Mean Log Loss", ascending=True).reset_index(drop=True)
brier_ranking = aggregated_results.sort_values("Mean Brier Score", ascending=True).reset_index(drop=True)

print("\nTop 5 Mean ROC-AUC ranked highest to lowest")
print(roc_auc_ranking.head(5).to_string(index=False))

print("\nTop 5 Mean AP ranked highest to lowest")
print(ap_ranking.head(5).to_string(index=False))

print("\nTop 5 Mean Log Loss ranked highest to lowest")
print(log_loss_ranking.head(5).to_string(index=False))

print("\nTop 5 Mean Brier Score ranked highest to lowest")
print(brier_ranking.head(5).to_string(index=False))

roc_auc_ranking.to_csv("outputs/ranking_ap.csv", index=False)
ap_ranking.to_csv("outputs/ranking_ap.csv", index=False)
log_loss_ranking.to_csv("outputs/ranking_log_loss.csv", index=False)
brier_ranking.to_csv("outputs/ranking_brier.csv", index=False)

Number of layers included
SSL Model
HuBERT      13
Wav2Vec2    13
WavLM       13
XLS-R       25
dtype: int64

Top 5 Mean ROC-AUC ranked highest to lowest
SSL Model  Layer  Mean ROC-AUC  Mean AP  Mean Log Loss  Mean Brier Score
    XLS-R     14      0.814220 0.488618       0.349000          0.112472
    XLS-R     15      0.812106 0.486715       0.352039          0.113729
    XLS-R     13      0.811193 0.480694       0.350092          0.112986
    XLS-R     16      0.810167 0.479786       0.355207          0.115090
    XLS-R     12      0.809288 0.480183       0.349917          0.112870

Top 5 Mean AP ranked highest to lowest
SSL Model  Layer  Mean ROC-AUC  Mean AP  Mean Log Loss  Mean Brier Score
    XLS-R     14      0.814220 0.488618       0.349000          0.112472
    XLS-R     15      0.812106 0.486715       0.352039          0.113729
    XLS-R     13      0.811193 0.480694       0.350092          0.112986
    XLS-R     12      0.809288 0.480183       0.349917          0.112870
   